In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("Reviews.csv")

# Show basic info
df.shape

(122155, 10)

In [2]:
df.columns


Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [3]:
df["Text"].head(2)


,Text
0,I have bought several of the Vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...


In [4]:
# Keep only relevant columns
df_clean = df[["Score", "Text"]].copy()

# Remove rows with missing text
df_clean.dropna(subset=["Text"], inplace=True)

# Check shape after cleaning
df_clean.shape


(122155, 2)

In [5]:
df_clean["Text"] = df_clean["Text"].str.lower()


In [6]:
import re

df_clean["Text"] = df_clean["Text"].apply(
    lambda x: re.sub(r"[^a-z\s]", "", x)
)


In [7]:
df_clean["Text"].head(2)


,Text
0,i have bought several of the vitality canned d...
1,product arrived labeled as jumbo salted peanut...


In [8]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

df_clean["Text"] = df_clean["Text"].apply(
    lambda x: " ".join([w for w in x.split() if w not in stop_words])
)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
df_clean["Text"].head(2)


,Text
0,bought several vitality canned dog food produc...
1,product arrived labeled jumbo salted peanutsth...


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# Use a sample to keep it fast for now
sample_texts = df_clean["Text"].sample(20000, random_state=42)

bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(sample_texts)

X_bow.shape


(20000, 5000)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(sample_texts)

X_tfidf.shape


(20000, 5000)

In [12]:
bow_vectorizer.get_feature_names_out()[:20]


array(['ability', 'able', 'abr', 'absolute', 'absolutely', 'absorb',
       'absorbed', 'acai', 'accept', 'acceptable', 'accepted', 'access',
       'accident', 'accidentally', 'according', 'account', 'accurate',
       'accustomed', 'acid', 'acidic'], dtype=object)

In [13]:
tfidf_vectorizer.get_feature_names_out()[:20]


array(['ability', 'able', 'abr', 'absolute', 'absolutely', 'absorb',
       'absorbed', 'acai', 'accept', 'acceptable', 'accepted', 'access',
       'accident', 'accidentally', 'according', 'account', 'accurate',
       'accustomed', 'acid', 'acidic'], dtype=object)

In [14]:
X_bow.shape


(20000, 5000)

In [15]:
X_tfidf.shape


(20000, 5000)

In [16]:
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# Train LDA model
lda = LatentDirichletAllocation(
    n_components=8,          # number of topics (we can adjust later)
    random_state=42,
    learning_method="batch"
)
lda.fit(X_bow)

# Helper: print top words per topic
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features_idx = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_features_idx]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

feature_names_bow = bow_vectorizer.get_feature_names_out()

print("LDA Topics (BoW):")
print_top_words(lda, feature_names_bow, n_top_words=10)


LDA Topics (BoW):
Topic 1: food, br, cat, cats, chicken, like, one, good, eat, foods
Topic 2: like, taste, flavor, chocolate, good, sweet, really, br, one, would
Topic 3: coffee, cup, like, good, flavor, br, one, strong, taste, blend
Topic 4: product, amazon, great, price, good, find, order, store, buy, time
Topic 5: dog, br, dogs, treats, one, food, get, like, would, treat
Topic 6: tea, green, drink, great, one, day, teas, love, good, like
Topic 7: chips, great, salt, like, good, br, flavor, taste, snack, love
Topic 8: br, sugar, water, like, oil, taste, product, use, coconut, drink


In [17]:
from sklearn.decomposition import NMF

nmf = NMF(
    n_components=8,     # same number of topics to compare fairly
    random_state=42
)
W = nmf.fit_transform(X_tfidf)

feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()

print("NMF Topics (TF-IDF):")
print_top_words(nmf, feature_names_tfidf, n_top_words=10)


NMF Topics (TF-IDF):
Topic 1: br, like, taste, good, flavor, really, one, dont, would, much
Topic 2: coffee, cup, strong, bold, roast, keurig, kcups, flavor, blend, like
Topic 3: tea, green, teas, drink, iced, flavor, chai, bags, black, love
Topic 4: dog, treats, dogs, loves, treat, teeth, one, love, give, small
Topic 5: great, product, price, amazon, find, love, buy, store, order, time
Topic 6: chips, salt, potato, bag, flavor, kettle, chip, vinegar, love, snack
Topic 7: chocolate, bar, bars, dark, cookies, love, milk, butter, peanut, hot
Topic 8: food, cat, cats, eat, foods, dry, chicken, canned, diet, baby
